# Métodos de Limit State Space para verificación de FCMs

Este notebook compara siete métodos de acotamiento por *limit state space* para la verificación de Mapas Cognitivos Difusos (FCMs), todos implementados con PyTorch:

1. **Naive** — iteración con aritmética de intervalos simple (`interval_state_space`).
2. **Symbolic** — propagación simbólica pura con relajaciones lineales (`symbolic_state_space`).
3. **Adaptive (naive + symbolic)** — arranque naive con `naive_iters=5` y luego simbólico (`warm_start_symbolic_state_space`).
4. **Rectified** — método combinado con calentamiento simbólico mínimo de 5 iteraciones (`containment_guided_state_space`).
5. **Fixed Reset** — reinicio fijo en `t=9` con 5 iteraciones naive de calentamiento (`periodic_reset_state_space`).
6. **Iterative Fixed Reset** — variante con 15 iteraciones naive de calentamiento (`iterative_periodic_reset_state_space`).
7. **Rectified Intersection** — rectificado con intersección de estados (`rectified_intersection_state_space`).

## Parámetros del experimento

- `n_concepts = 12`, `density = 0.7`, `activation = "sigmoid"`, `phi = 0.7`
- `MAX_ITERS = 50`, `n_samples = 100`, `l_0 = 0`, `u_0 = 1`, `ADAPTIVE_THRESHOLD = 0.3`
- `SEED = 16607`; la matriz de influencia `W` se genera con `generate_random_matrix` y se usa su transpuesta `Wt`.

Cada celda de graficado compara la evolución de los límites del método de interés (línea 2) contra los estados naive (línea 1), con 100 trayectorias simuladas superpuestas (`compare_state_evolutions_and_simulations`). Las celdas finales imprimen el *covering* de cada método y verifican si las muestras caen dentro de los estados calculados.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float64
torch.set_grad_enabled(False)
torch.set_default_dtype(dtype)
torch.set_printoptions(precision=30, sci_mode=False)

print(f"Device: {device}")

In [ ]:
from typing import Literal

ActivationFunction = Literal["sigmoid", "tanh"]

In [ ]:
import numpy as np
from core.extras import rescaled_reasoning
from core.methods import (
    interval_state_space,
    warm_start_symbolic_state_space,
    rectified_intersection_state_space,
    containment_guided_state_space,
    symbolic_state_space,
)
from core.metrics import covering, is_inside_state_series, state_series_from_samples
from core.symbolic import concretize_state_space
from core.utils import generate_input_samples, generate_inputs, generate_random_matrix
from experimental.methods import (
    periodic_reset_state_space,
    iterative_periodic_reset_state_space,
)
from luis.plot import compare_state_evolutions_and_simulations


MAX_ITERS = 50

n_concepts = 12
density = 0.7
activation: ActivationFunction = "sigmoid"  # "tanh"

phi = 0.7
n_samples = 100
l_0 = 0
u_0 = 1
ADAPTIVE_THRESHOLD = 0.3


seed = int(np.random.randint(0, int(1e5)))
# seed = 36850
# seed = 83907
# seed = 9066
# seed = 36201
# seed = 16607

print(f"SEED = {seed}")

np.random.seed(seed)
torch.manual_seed(seed)

W = generate_random_matrix(n_concepts, density, device=device, dtype=dtype)

Wt = W.T
inputs = generate_inputs(n_concepts, l_0, u_0, device=device).to(dtype=dtype)

naive_states = interval_state_space(
    Wt,
    inputs,
    max_iters=MAX_ITERS,
    phi=phi,
    activation_function=activation,
    device=device,
)

A = generate_input_samples(n_samples, n_concepts, l_0, u_0, device=device)

symbolic_states = symbolic_state_space(
    Wt,
    inputs,
    activation,
    max_iters=MAX_ITERS,
    phi=phi,
    overapprox_threshold=None,
    device=device,
)

concrete_states = concretize_state_space(symbolic_states, inputs, device=device)


symbolic_adaptive_states = warm_start_symbolic_state_space(
    Wt,
    inputs,
    activation,
    phi=phi,
    max_iters=MAX_ITERS,
    device=device,
    arithmetic_iters=5,
)

adaptive_states = concretize_state_space(
    symbolic_adaptive_states,
    inputs,
)

simulated_states = rescaled_reasoning(
    A,
    W,
    T=MAX_ITERS,
    phi=phi,
    transfer_function=torch.sigmoid if activation == "sigmoid" else torch.tanh,
    device=device,
)

sampled_states = state_series_from_samples(simulated_states)

fixed_reset_states = periodic_reset_state_space(
    Wt,
    inputs,
    activation,
    phi=phi,
    t=9,
    max_iters=MAX_ITERS,
    arithmetic_warming_iters=5,
    baseline_interval_series=naive_states,
    device=device,
)

rectified_states = containment_guided_state_space(
    Wt,
    inputs,
    activation,
    phi=phi,
    max_iters=MAX_ITERS,
    minimal_symbolic_warming_iters=5,
    baseline_interval_series=naive_states,
)

iterative_fixed_reset_states = iterative_periodic_reset_state_space(
    Wt,
    inputs,
    activation,
    phi=phi,
    max_iters=MAX_ITERS,
    arithmetic_warming_iters=15,
    baseline_interval_series=naive_states,
    device=device,
)

rectified_intersection_states = rectified_intersection_state_space(
    Wt,
    inputs,
    activation,
    phi,
    MAX_ITERS,
    minimal_warming_iters=5,
    baseline_interval_series=naive_states,
)

print(
    f"Samples lie inside rectified states: {is_inside_state_series(sampled_states, fixed_reset_states, tolerance=1e-6)}"
)


print(f"Is inside symbolic: {is_inside_state_series(sampled_states, concrete_states)}")
print(f"Is inside adaptive: {is_inside_state_series(sampled_states, adaptive_states)}")
print(
    f"Is inside rectified: {is_inside_state_series(sampled_states, fixed_reset_states)}"
)

print(f"Naive Covering: {covering(naive_states)}")
print(f"Symbolic Covering: {covering(concrete_states)}")
print(f"Adaptive Covering: {covering(adaptive_states)}")
print(f"Rectified Covering: {covering(rectified_states)}")
print(f"Fixed Reset Covering: {covering(fixed_reset_states)}")
print(f"Rectified Intersection Covering: {covering(rectified_intersection_states)}")
print(f"Iterative Fixed Reset Covering: {covering(iterative_fixed_reset_states)}")

## Explicación de los métodos

Todos los métodos sobreaproximan el conjunto de estados alcanzables de la FCM; se diferencian en cómo manejan la **doble no linealidad** (activación $\phi(\cdot)$ + residuo $\phi$) y en el coste frente a la precisión (anchura) de los intervalos resultantes.

### Métodos con aritmética de intervalos pura

- **`interval_state_space` (Naive)** — Aritmética de intervalos clásica. Descompone la matriz de pesos en parte positiva y negativa ($W_+$, $W_-$) para tener en cuenta el signo, redondea siempre hacia afuera y aplica activación + residual paso a paso. Repite el esquema hasta `max_iters`. Es el más barato y el punto de partida de los demás; su coste es que produce intervalos anchos (sobreaproximación grande).

### Métodos simbólicos (con relajación de la activación)

- **`symbolic_state_space`** — En vez de concretizar en cada paso, mantiene cada estado como una **combinación lineal simbólica de los inputs iniciales** y la propaga a través de $W$. Solo al final se concreta y se aplica la activación mediante **relajación lineal** (tangente/secante). Si la sobreaproximación de una entrada supera un umbral (`overapprox_threshold`), la reemplaza por su valor concreto para no arrastrar el error. Más preciso que Naive, pero más caro.
- **`warm_start_symbolic_state_space` (Adaptive)** — Primero ejecuta `naive_iters` pasos del método Naive y a partir de ahí cambia a simbólico. Se beneficia del bajo coste del Naive en las primeras iteraciones y gana precisión con el simbólico después.

### Métodos rectificados / de intersección

- **`containment_guided_state_space`** — Estrategia rectificada: alterna pasos simbólicos y Naive mientras haya **contención** entre ambos; cuando la pierde, hace `naive_rectifications` pasos Naive para recuperar estabilidad.
- **`rectified_intersection_state_space`** — Ejecuta `containment_guided_state_space` varias veces (típicamente 10) y **refina la intersección acumulada** de todas las pasadas como baseline. Es el método de referencia en este notebook.
- **`periodic_reset_state_space` (experimental)** — Simbólico con **reinicio periódico** de las variables en el instante `t`: en cada periodo re-emite un subconjunto de estados para limitar la acumulación de error.
- **`iterative_periodic_reset_state_space` (experimental)** — Encadena reinicios con `t ∈ {-1, 1..15}` e **interseca las pasadas**, intentando combinar la precisión del simbólico con la estabilidad del reinicio.

### Simulación de referencia

- **`rescaled_reasoning`** — No es una verificación: simula la dinámica real de la FCM a partir de muestras de entrada ($H[t] = \phi \cdot f(H[t-1]\,W) + (1-\phi)\,A_0$) y sirve como **ground truth** para comparar si los estados verificados contienen de verdad los estados simulados.

In [ ]:
compare_state_evolutions_and_simulations(
    naive_states.cpu().numpy(),
    iterative_fixed_reset_states.cpu().numpy(),
    simulated_states.cpu().numpy(),
    "Naive",
    "Iterative Fixed",
)

In [ ]:
compare_state_evolutions_and_simulations(
    naive_states.cpu().numpy(),
    fixed_reset_states.cpu().numpy(),
    simulated_states.cpu().numpy(),
    "Naive",
    "New Method",
)

In [ ]:
compare_state_evolutions_and_simulations(
    naive_states.cpu().numpy(),
    rectified_states.cpu().numpy(),
    simulated_states.cpu().numpy(),
    "Naive",
    "Rectified",
)

In [ ]:
compare_state_evolutions_and_simulations(
    naive_states.cpu().numpy(),
    rectified_intersection_states.cpu().numpy(),
    simulated_states.cpu().numpy(),
    "Naive",
    "Rectified",
)